# Camada Silver: Transformação e Normalização de Dados

Neste notebook, faremos o processamento da tabela `matches` extraída na Camada Bronze.
O objetivo é isolar a linha do tempo dos Gols em uma tabela própria para permitir a análise da **Eficiência Emocional** e facilitar a orquestração de cargas incrementais (via Databricks Workflows ou Apache Airflow).

In [ ]:
# --- CONFIGURAÇÃO AGNÓSTICA DO AMBIENTE ---
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.appName("FootballPipeline_Silver").getOrCreate()

ENVIRONMENT = os.getenv("SPARK_ENV", "databricks")

if ENVIRONMENT == "databricks":
    PATH_BRONZE = "/Volumes/workspace/bronze/football_raw"
    PATH_SILVER = "/Volumes/workspace/silver/football_processed"
    CATALOG_NAME = "workspace"
else:
    PATH_BRONZE = "./data/bronze"
    PATH_SILVER = "./data/silver"
    CATALOG_NAME = "local"

print(f"Ambiente Configurado: {ENVIRONMENT}")

In [ ]:
# 1. Leitura da Camada Bronze
if ENVIRONMENT == "databricks":
    df_matches = spark.table(f"{CATALOG_NAME}.bronze.matches")
else:
    df_matches = spark.read.format("delta").load(f"{PATH_BRONZE}/matches")

print(f"Total de partidas na Bronze: {df_matches.count()}")
df_matches.show(5)

In [ ]:
# 2. Criação da tabela de Partidas Limpas (Sem os arrays pesados de gols)
df_silver_matches = df_matches.drop("goals_home", "goals_away")

if ENVIRONMENT == "databricks":
    df_silver_matches.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG_NAME}.silver.matches")
else:
    df_silver_matches.write.format("delta").mode("overwrite").save(f"{PATH_SILVER}/matches")

print("Tabela silver.matches salva com sucesso!")

In [ ]:
 Linha do Tempo dos Gols 

# Extraindo os gols do time da casa (Team 1)
df_goals_home = df_matches.select(
    F.col("match_id"),
    F.col("team1").alias("scoring_team"),
    F.col("team2").alias("conceding_team"),
    F.explode_outer(F.col("goals_home")).alias("goal")
).filter(F.col("goal").isNotNull())

# Extraindo os gols do time visitante (Team 2)
df_goals_away = df_matches.select(
    F.col("match_id"),
    F.col("team2").alias("scoring_team"),
    F.col("team1").alias("conceding_team"),
    F.explode_outer(F.col("goals_away")).alias("goal")
).filter(F.col("goal").isNotNull())

# Juntando as duas pontas em uma tabela só (Union)
df_all_goals = df_goals_home.union(df_goals_away)

# "Desempacotando" as colunas de dentro do JSON do gol
df_silver_goal_events = df_all_goals.select(
    F.col("match_id"),
    F.col("scoring_team"),
    F.col("conceding_team"),
    F.col("goal.name").alias("scorer"),
    F.col("goal.minute").alias("minute"),
    F.col("goal.penalty").alias("is_penalty"),
    F.col("goal.owngoal").alias("is_owngoal")
)

df_silver_goal_events.show(10)

In [ ]:
# 4. Salvando a Tabela de Eventos (Nossa base analítica)
if ENVIRONMENT == "databricks":
    df_silver_goal_events.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG_NAME}.silver.goal_events")
else:
    df_silver_goal_events.write.format("delta").mode("overwrite").save(f"{PATH_SILVER}/goal_events")

print("Tabela silver.goal_events salva com sucesso! Pronta para a Camada Gold.")